In [3]:
import requests

import pandas as pd

# Explainer Notebook - Final Project
02806 Social Data Analysis and Visualization - Spring 2026 - Group 42 

## The Basis of Analysis

The aim of this analysis is to showcase if socioeconomic hardship is correlated to a rise in criminal activity. The hypothesis is, that a areas with a higher hardship will have a tendency to a higher crime rate, specifically in the sector of "blue-collar" crime (aka street crime). To make that analysis we will be zooming in on the city of Chicago, US, as an example. We are employing two data sets from, both collected by the city itself, to explore the possible connection between crime and socioeconomic status. This should be seen as a visual journey, i.e. there wont be a focus on specific numbers, but rather on qualitative analysis of various data points.

## The Data Sets

### Socioeconomic data

The first data set for this project is the Census data set collected by the city of Chicago. The aim of this data set is to document and explore the tendencies of poverty, employment, age, and education in the different communities of Chicago. In the data set we also see a custom "Hardship" score, which is an aggregation of the other parameters normalized by the area of the community.

Now, let's have a look at a sample of the data

In [2]:
sd = pd.read_csv('./data/sociodata.csv')
sd.sample(frac=1).reset_index(drop=True).head(5)

,Community Area Number,COMMUNITY AREA NAME,PERCENT OF HOUSING CROWDED,PERCENT HOUSEHOLDS BELOW POVERTY,PERCENT AGED 16+ UNEMPLOYED,PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA,PERCENT AGED UNDER 18 OR OVER 64,PER CAPITA INCOME,HARDSHIP INDEX
0,19.0,Belmont Cragin,10.8,18.7,14.6,37.3,37.3,15461,70.0
1,1.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
2,73.0,Washington Height,1.1,16.9,20.8,13.7,42.6,19713,48.0
3,47.0,Burnside,6.8,33.0,18.6,19.3,42.7,12515,79.0
4,21.0,Avondale,6.0,15.3,9.2,24.7,31.0,20039,42.0


As seen in the example we have data about the various socioeconomic, factor explained previously, as well as the geographic community area. This gives an obvious opportunity to visualize the data as a choropleth plot. This visualization will allow us to analyze where, in geographical terms, there is high "hardship" in the city, as well as easily look at correlations between the different factors.

In [ ]:
def _get_shapes_file() -> dict:
    """Return geojson for chicago community areas"""
    # import shapefile for community areas
    shapefile = requests.get("https://data.cityofchicago.org/resource/igwz-8jzy.json").json()

    # And convert them to proper geojson format
    geojson = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "geometry": row["the_geom"],
                "properties": {k: v for k, v in row.items() if k != "the_geom"},
            }
            for row in shapefile
            if row.get("the_geom") and row["the_geom"].get("coordinates")
        ],
    }
    return geojson

def _align_naming_schemes(df: pd.DataFrame) -> pd.DataFrame:
    """Align the naming schemes of the comminity areas (city of chicago misspelled some names in the census data... smh)"""
    df = df.copy()
    # Spell correction
    name_fixes = {
        "MONTCLAIRE": "MONTCLARE",
        "WASHINGTON HEIGHT": "WASHINGTON HEIGHTS",
        "O'HARE": "OHARE",
    }
    df["COMMUNITY AREA NAME"] = df["COMMUNITY AREA NAME"].str.upper().replace(name_fixes)
    # Exclude the "total" row
    df = df[df["COMMUNITY AREA NAME"] != "CHICAGO"]
    return df

def plot_income_choropleth(col: str, filename: str, plot_data: pd.DataFrame, geojson: dict) -> None:
    import plotly.express as px
    import plotly.graph_objects as go

    fig = go.Figure(
        data=px.choropleth_map(
            plot_data,
            geojson=geojson,
            locations="COMMUNITY AREA NAME",
            color=col,
            featureidkey="properties.community",
            color_continuous_scale="Magma",
            map_style="carto-positron",
            zoom=9,
            center={"lat": 41.8781, "lon": -87.6298},
            opacity=0.6,
        )
    )
    fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
    fig.write_html(f"./../docs/figures/{filename}", include_plotlyjs="cdn")

geojson = _get_shapes_file()
plot_data = _align_naming_schemes(sd)

# Generate choropleth plots for the 4 main columns
plot_income_choropleth("HARDSHIP INDEX", "hardship_choropleth.html", plot_data, geojson)
plot_income_choropleth("PER CAPITA INCOME ", "income_choropleth.html", plot_data, geojson) # Yes, trailing space is on purpose... smh
plot_income_choropleth("PERCENT HOUSEHOLDS BELOW POVERTY", "poverty_choropleth.html", plot_data, geojson)
plot_income_choropleth("PERCENT AGED 16+ UNEMPLOYED", "unemployment_choropleth.html", plot_data, geojson)